In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

from monai.transforms import (
    LoadImage, EnsureChannelFirst, Compose, Resize, NormalizeIntensity, Lambda, ToTensor
)
from monai.utils import set_determinism
from monai.config import print_config
from monai.metrics import ROCAUCMetric
from monai.transforms import Activations, AsDiscrete
from monai.data import decollate_batch

In [ ]:
set_seed_val = 42
set_determinism(seed=set_seed_val)
random.seed(set_seed_val)
np.random.seed(set_seed_val)
torch.manual_seed(set_seed_val)

print_config()

In [ ]:
base_dir = 'dataset'
model_save_path = './models/effv2s_best.pth'

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (M1 GPU acceleration)")
else:
    device = torch.device('cpu')
    print("MPS not available, using CPU")

In [ ]:
IMAGE_SIZE = 224                 
BATCH_SIZE = 4
EPOCHS = 60 # Resultados pararam de melhorar depois de 12 epochs
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
NUM_WORKERS = 0

In [ ]:
phases = ['train', 'val', 'test']
data = {phase: {'images': [], 'labels': []} for phase in phases}
class_names = sorted([x for x in os.listdir(os.path.join(base_dir, 'train')) if os.path.isdir(os.path.join(base_dir, 'train', x))])
num_class = len(class_names)

def load_images_labels(phase):
    for i, class_name in enumerate(class_names):
        class_dir = os.path.join(base_dir, phase, class_name)
        valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        if not os.path.isdir(class_dir):
            continue
        image_files = [os.path.join(class_dir, x) for x in os.listdir(class_dir) if x.lower().endswith(valid_exts)]
        data[phase]['images'].extend(image_files)
        data[phase]['labels'].extend([i] * len(image_files))

for phase in phases:
    load_images_labels(phase)
    print(f"{phase} count = {len(data[phase]['images'])}")

In [ ]:
def repeat_if_needed(img):
    if img.shape[0] == 1:
        return img.repeat(3, 1, 1)
    return img

from monai.transforms import (
    RandFlip, RandRotate, RandAffine, RandZoom, RandAdjustContrast, RandGaussianNoise, RandGaussianSmooth, RandScaleIntensity, RandBiasField
)
train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((IMAGE_SIZE, IMAGE_SIZE)),    
    RandRotate(range_x=8, prob=0.3), 
    RandZoom(min_zoom=0.95, max_zoom=1.05, prob=0.2),
    RandAdjustContrast(prob=0.3, gamma=(0.9, 1.3)),
    RandGaussianNoise(prob=0.2, std=0.01),
    RandGaussianSmooth(prob=0.2, sigma_x=(0.5, 1.5)),
    RandScaleIntensity(prob=0.2, factors=0.1),
    
    NormalizeIntensity(),
    Lambda(repeat_if_needed),
    ToTensor()
])

val_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    NormalizeIntensity(),
    Lambda(repeat_if_needed),
    ToTensor()
])

In [ ]:
# Escolher uma imagem (primeira da primeira classe)
img_path = os.path.join(base_dir, 'train', class_names[0], 
                        os.listdir(os.path.join(base_dir, 'train', class_names[0]))[0])
print(f"Imagem: {img_path}")

# Carregar imagem original (sem augmentation)
original_pil = Image.open(img_path)
original_np = np.array(original_pil)

# Aplicar transformações
augmented_tensor = train_transforms(img_path)
augmented_np = augmented_tensor.permute(1, 2, 0).cpu().numpy()

# Desnormalizar para visualização
if augmented_np.max() <= 1.0:
    augmented_np = (augmented_np * 255).astype(np.uint8)
else:
    augmented_np = ((augmented_np - augmented_np.min()) / 
                    (augmented_np.max() - augmented_np.min()) * 255).astype(np.uint8)

# Visualizar no mesmo plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(original_np)
plt.title(f'Original\n{class_names[0]}', fontsize=14, fontweight='bold')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(augmented_np)
plt.title('Com Augmentation\n(Rotate + Zoom + Contrast + Noise)', fontsize=14, fontweight='bold')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class ERCPDataset(Dataset):
    def __init__(self, image_files, labels, transforms):
        self.image_files = image_files
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img_path = self.image_files[index]
        img = self.transforms(img_path)
        lbl = self.labels[index]
        return img, lbl

train_ds = ERCPDataset(data['train']['images'], data['train']['labels'], train_transforms)
val_ds = ERCPDataset(data['val']['images'], data['val']['labels'], val_transforms)
test_ds = ERCPDataset(data['test']['images'], data['test']['labels'], val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

In [ ]:
import torchvision.models as tm

weights = tm.EfficientNet_V2_S_Weights.IMAGENET1K_V1
base_model = tm.efficientnet_v2_s(weights=weights)

in_features = base_model.classifier[1].in_features if isinstance(base_model.classifier, nn.Sequential) else base_model.classifier.in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_class)
)
model = base_model.to(device)

In [ ]:
# Class weighting to help imbalance
class_counts = np.bincount(data['train']['labels'], minlength=num_class).astype(np.float32)
class_weights = (1.0 / (class_counts + 1e-9))
class_weights = class_weights / class_weights.sum() * num_class
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=0.05)  # label smoothing helps generalization
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

# Mixed precision scaler
scaler = GradScaler()

# Metrics helpers
act = Activations(softmax=True)
to_onehot = AsDiscrete(to_onehot=num_class)
auc_metric = ROCAUCMetric()

In [ ]:
best_val_f1 = -1.0
epochs_without_improvement = 0

def compute_metrics_from_outputs(all_logits, all_labels):
    # all_logits: torch tensor (N, num_class), all_labels: torch tensor (N,)
    preds = torch.argmax(all_logits, dim=1).detach().cpu().numpy()
    labels = all_labels.detach().cpu().numpy()
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return f1, acc, preds, labels

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    all_train_logits = []
    all_train_labels = []

    t0 = time.perf_counter()
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        # gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.detach().item() * imgs.size(0)
        all_train_logits.append(logits.detach())
        all_train_labels.append(labels.detach())

    scheduler.step(epoch + (epoch % 1) / 1)  # step scheduler (works with WarmRestarts)
    epoch_loss = epoch_loss / len(train_loader.dataset)
    train_logits = torch.cat(all_train_logits, dim=0)
    train_labels = torch.cat(all_train_labels, dim=0)
    train_f1, train_acc, _, _ = compute_metrics_from_outputs(train_logits, train_labels)

    # Validation
    model.eval()
    all_val_logits = []
    all_val_labels = []
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            with autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
            val_loss += loss.detach().item() * imgs.size(0)
            all_val_logits.append(logits.detach())
            all_val_labels.append(labels.detach())

    val_loss = val_loss / len(val_loader.dataset)
    val_logits = torch.cat(all_val_logits, dim=0)
    val_labels = torch.cat(all_val_labels, dim=0)
    val_f1, val_acc, val_preds_flat, val_labels_flat = compute_metrics_from_outputs(val_logits, val_labels)

    # AUC (MONAI)
    y_onehot = [to_onehot(i) for i in decollate_batch(val_labels, detach=False)]
    y_pred_act = [act(i) for i in decollate_batch(val_logits)]
    auc_metric(y_pred_act, y_onehot)
    auc_result = auc_metric.aggregate()
    auc_metric.reset()

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss: {epoch_loss:.4f} train_f1: {train_f1:.4f} train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} val_f1: {val_f1:.4f} val_acc: {val_acc:.4f} val_auc: {auc_result:.4f} "
          f"time: {time.perf_counter()-t0:.1f}s")

    # checkpoint on val_f1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), model_save_path)
        print(f"Saved new best model (val_f1={best_val_f1:.4f})")
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}. Best val_f1={best_val_f1:.4f}")
        break

print("Training finished")

In [ ]:
model.load_state_dict(torch.load(model_save_path))
model.eval()

def evaluate_test(loader, model):
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu().numpy().reshape(-1,1))
            all_labels.append(labels.cpu().numpy().reshape(-1,1))
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc:.4f} | Test F1 (macro): {f1:.4f}")
    return all_labels, all_preds

test_labels_arr, test_preds_arr = evaluate_test(test_loader, model)

# Classification report + confusion matrix
print(classification_report(test_labels_arr, test_preds_arr, target_names=class_names, digits=4, zero_division=0))

cm = confusion_matrix(test_labels_arr, test_preds_arr)
plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='g', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix')
plt.tight_layout()
plt.savefig('effv2s_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
print(cm)